# Self-Attention in a Transformer Encoder
### Part 2 – Transformer Question Answering Task

This notebook implements the **self-attention mechanism** from scratch using NumPy, applied to the input sentence:

> *"What are the symptoms of diabetes?"*

The steps follow the standard Transformer encoder attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

## Step 1 – Import Libraries

In [ ]:
import numpy as np

## Step 2 – Define the Softmax Function

In [ ]:
def softmax(x):
    """
    Numerically stable softmax applied row-wise.
    Subtracting the row max prevents overflow for large dot products.
    """
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

## Step 3 – Define the Self-Attention Function

In [ ]:
def self_attention(sentence, d_model=8, d_k=4):
    """
    Computes self-attention scores for a tokenized sentence.

    Parameters:
    -----------
    sentence : list of str
        Tokenized input sentence.
    d_model : int
        Dimensionality of token embeddings (and output).
    d_k : int
        Dimensionality of Query and Key projections.

    Returns:
    --------
    attention_weights : ndarray (seq_len x seq_len)
        Softmax attention distribution for each token.
    output : ndarray (seq_len x d_model)
        Weighted sum of Value vectors.
    """
    seq_len = len(sentence)
    np.random.seed(42)  # For reproducibility

    # ── Step 3a: Token Embeddings ──────────────────────────────────────────
    # In a real model these are learned; here we use random vectors.
    embeddings = np.random.randn(seq_len, d_model)
    print(f"Input sentence : {sentence}")
    print(f"Embeddings shape: {embeddings.shape}  ({seq_len} tokens × {d_model} dims)\n")

    # ── Step 3b: Weight Matrices (learned during training) ─────────────────
    W_Q = np.random.randn(d_model, d_k)
    W_K = np.random.randn(d_model, d_k)
    W_V = np.random.randn(d_model, d_model)

    # ── Step 3c: Compute Q, K, V ───────────────────────────────────────────
    Q = embeddings @ W_Q   # (seq_len × d_k)
    K = embeddings @ W_K   # (seq_len × d_k)
    V = embeddings @ W_V   # (seq_len × d_model)

    # ── Step 3d: Scaled Dot-Product Attention Scores ───────────────────────
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_len × seq_len)
    print("Raw Attention Scores  (Q · Kᵀ / √d_k):")
    print(np.round(scores, 3), "\n")

    # ── Step 3e: Softmax → Attention Weights ──────────────────────────────
    attention_weights = softmax(scores)   # (seq_len × seq_len)
    print("Attention Weights (after softmax):")
    print(np.round(attention_weights, 3), "\n")

    # ── Step 3f: Weighted Sum of Values ───────────────────────────────────
    output = attention_weights @ V   # (seq_len × d_model)
    print(f"Attention Output shape: {output.shape}\n")

    return attention_weights, output

## Step 4 – Run on the Task Sentence

In [ ]:
sentence = ["What", "are", "the", "symptoms", "of", "diabetes", "?"]

weights, output = self_attention(sentence)

## Step 5 – Inspect Which Token Each Word Attends to Most

In [ ]:
print("Top attention focus per token:")
print("-" * 45)
for i, token in enumerate(sentence):
    top_j = np.argmax(weights[i])
    print(f"  '{token}' attends most to '{sentence[top_j]}' "
          f"(score: {weights[i, top_j]:.3f})")

## Step 6 – Visualize the Attention Heatmap

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(weights, cmap="Blues")

ax.set_xticks(range(len(sentence)))
ax.set_yticks(range(len(sentence)))
ax.set_xticklabels(sentence, rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(sentence, fontsize=11)

# Annotate each cell with its weight value
for i in range(len(sentence)):
    for j in range(len(sentence)):
        ax.text(j, i, f"{weights[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if weights[i, j] > 0.25 else "black")

ax.set_title("Self-Attention Weights\n'What are the symptoms of diabetes?'",
             fontsize=12, pad=12)
ax.set_xlabel("Keys (attended to)", fontsize=10)
ax.set_ylabel("Queries (attending)", fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()